In [3]:
import qim3d

# Generate tubular synthetic blob
vol = qim3d.generate.volume(base_shape = (10, 300, 300),
                        final_shape = (100, 100, 100),
                        noise_scale = 0.0003,
                        gamma = 2,
                        threshold = 0.0,
                        volume_shape = "cylinder",
                        old=True
                        )

# Visualize synthetic volume
qim3d.viz.volumetric(vol)

Output()

In [14]:
import numpy as np

base_shape = (256, 128, 128)

# Generate grid of coordinates
z, y, x = np.indices(base_shape)

# Calculate the distance from the center of the shape
center = np.array(base_shape) / 2

# Normalized coordinates
dx = (x - center[2]) / center[2]
dy = (y - center[1]) / center[1]
dz = (z - center[0]) / center[0]

# Normal ellipsoidal distance
dist = np.sqrt(dx**2 + dy**2 + dz**2)

dist = np.power(dist, 4)
# Clip them and normalize
dist = np.clip(dist, 0, 1.2) / 1.2

qim3d.viz.volumetric(1-dist, grid_visible=True)

Output()

In [19]:
base_shape = (256, 128, 128)
rounding_radius = 0.2  # As a fraction of half-size

# Generate normalized coordinates from -1 to 1
z, y, x = np.indices(base_shape)
center = np.array(base_shape) / 2
half_size = np.array(base_shape) / 2

# Normalized coordinate grid [-1, 1]
nx = (x - center[2]) / half_size[2]
ny = (y - center[1]) / half_size[1]
nz = (z - center[0]) / half_size[0]

# Combine to a single array for easier math
p = np.stack([nx, ny, nz], axis=-1)

# Box half-extents (size of cube) in normalized coords
box_half_extent = np.array([0.8, 0.8, 0.8])  # controls cube size
radius = rounding_radius

# Signed distance to rounded box
q = np.abs(p) - box_half_extent
outside_dist = np.linalg.norm(np.maximum(q, 0), axis=-1)
inside_dist = np.minimum(np.maximum.reduce(q, axis=-1), 0)
sdf = outside_dist + inside_dist - radius

#sdf = (sdf - np.min(sdf)) / np.max(sdf) - np.min(sdf)

sdf_clipped = np.clip(sdf, None, 0)
sdf_normalized = sdf_clipped / np.min(sdf_clipped)
sdf_normalized = abs(1-sdf_normalized)  # values go from 0 (center) to 1 (edge), outside is >1
print(np.min(sdf_normalized))
print(np.max(sdf_normalized))
#sdf_normalized = np.clip(sdf_normalized, 0, 1)

qim3d.viz.volumetric(1-sdf_normalized, grid_visible=True)

0.0
1.0


Output()

In [ ]:
s = 25
vol_padded = qim3d.operations.pad(vol, x_axis=s, y_axis=s, z_axis=s)
strel = np.ones((s,s,s))

vol_closed = qim3d.morphology.closing(vol_padded, strel, method='ndi')
print(vol_closed.shape)
vol_trimmed = qim3d.operations.trim(vol_closed)
vol_final = qim3d.operations.pad_to(vol_trimmed, (128,128,128))
print(vol_final.shape)
qim3d.viz.volumetric(vol_final)

(178, 178, 178)
(128, 128, 128)


Output()

In [ ]:
s = 25
vol_padded = qim3d.operations.pad(vol, x_axis=s, y_axis=s, z_axis=s)
strel = np.ones((s,s,s))

vol_black = qim3d.morphology.black_tophat(vol_padded, strel, method='ndi')
vol_trimmed = qim3d.operations.trim(vol_black)
vol_black = qim3d.operations.pad_to(vol_trimmed, (128,128,128))

qim3d.viz.volumetric(vol_black)

Output()

In [ ]:
s = 10
strel = np.ones((s,s,s))

vol_opened = qim3d.morphology.opening(vol, strel, method='ndi')

qim3d.viz.volumetric(vol_opened)

Output()

In [ ]:
s = 10

strel = np.ones((s,s,s))

vol_white = qim3d.morphology.white_tophat(vol, strel, method='ndi')

qim3d.viz.volumetric(vol_white)

Output()

In [ ]:
s = 10
strel = np.ones((s,s,s))

vol_dilated = qim3d.morphology.dilate(vol, strel, method='ndi')

qim3d.viz.volumetric(vol_dilated)

Output()

In [ ]:
s = 5

strel = np.ones((s,s,s))

vol_eroded = qim3d.morphology.erode(vol, strel, method='ndi')

qim3d.viz.volumetric(vol_eroded)

Output()

In [8]:

# Apply noise to the synthetic collection
vol_n = qim3d.generate.background(
    background_shape = vol.shape,
    baseline_value = 0,
    min_noise_value = 0,
    max_noise_value = 30,
    generate_method = 'add',
    apply_method = 'divide',
    apply_to = vol
)
print(vol_n.shape)
#qim3d.viz.volumetric(noisy_collection)


(128, 128, 128)


In [ ]:
base_shape = (128, 128, 128)
z, y, x = np.indices(base_shape)

center = np.array(base_shape) / 2

dist = np.sqrt((z - center[0]) ** 2 + (y - center[1]) ** 2 + (x - center[2]) ** 2)

dist /= np.sqrt(3 * (center[0] ** 2))
dist = np.clip(dist, 0, 0.7071) / 0.7071
qim3d.viz.volumetric(dist)

Output()

In [ ]:
import scipy.ndimage as ndi
print(vol.shape)

zoomed = ndi.zoom(vol, 0.5)
print(zoomed.shape)

(128, 128, 128)
(64, 64, 64)
